# Model evaluation pipeline

This notebook runs the **decision-useful evaluation workflow** on holdout (test) predictions:

1. **Task detection** — regression vs classification
2. **Headline metrics** — MAE, RMSE, R² (regression)
3. **Core diagnostic plots** — predicted vs actual, residuals vs predicted, residual distribution
4. **Subgroup performance** — error by subgroup (e.g. state) when columns are available
5. **Time-based performance** — actual vs predicted over time when a time column exists
6. **Top error inspection** — largest absolute errors with key features
7. **Interpretation** — what the diagnostics suggest and recommended next steps

Uses **test-set predictions only** (no training evaluation).

## Setup and inputs

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_repo_root() -> Path:
    """Find repo root regardless of notebook working directory."""
    p = Path.cwd().resolve()
    for candidate in [p] + list(p.parents):
        # Case A: candidate is repo root => look for modeling/evaluation.py
        if (candidate / "modeling" / "model_evaluation.py").exists():
            return candidate
        # Case B: candidate is modeling/ => look for evaluation.py directly
        if (candidate / "model_evaluation.py").exists() and candidate.name == "modeling":
            return candidate.parent
    raise FileNotFoundError(
        "Could not find `modeling/model_evaluation.py` (repo root) or `model_evaluation.py` inside `modeling/`. "
        f"Searched upward starting from: {p}"
    )


REPO_ROOT = _find_repo_root()
# Ensure we can import evaluation.py regardless of notebook cwd.
sys.path.insert(0, str(REPO_ROOT / "modeling"))

from model_evaluation import evaluate, detect_task, regression_subgroup_stats
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "modeling" / "results"

# Which model's predictions to evaluate (file under results/<TARGET>/)
TARGET = "CASTHMA"
MODEL_NAME = "xgboost"  # e.g. ridge, lasso, elasticnet, xgboost

# Exposure set to evaluate (must match exported predictions)
EXPOSURE_KEY = "Full_pesticides_raw"  # set to "Aggs_engineered" or "Aggs_raw" if desired

PRED_PATH = RESULTS_DIR / TARGET / f"{MODEL_NAME}_predictions_{EXPOSURE_KEY}.csv"
if not PRED_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PRED_PATH.name}. "
        "Run `model_selection.ipynb` export cell (cell 18) to generate the suffixed exposure predictions. "
        f"Expected at: {PRED_PATH}."
    )

TEST_PATH = DATA_DIR / f"test_{TARGET}.csv"


## Load predictions and align test data

In [ ]:
pred_df = pd.read_csv(PRED_PATH)
print("Predictions:", pred_df.shape)
print(pred_df.head())

y_true = pred_df["actual"].values
y_pred = pred_df["prediction"].values

# Optional: merge with test set for subgroups and time
X_test = None
subgroup_columns = {}
time_column = None

def _qsplit(series, *, q=0.5, high_label="high", low_label="low", invert=False):
    s = pd.to_numeric(series, errors="coerce")
    thr = s.quantile(q)
    if invert:
        # e.g. lower income => higher poverty
        out = np.where(s <= thr, high_label, low_label)
    else:
        out = np.where(s >= thr, high_label, low_label)
    out[pd.isna(s)] = None
    return out

if TEST_PATH.exists():
    test_df = pd.read_csv(TEST_PATH)
    # Align by (FIPS, YEAR)
    merge_cols = [c for c in ["FIPS", "YEAR"] if c in pred_df.columns and c in test_df.columns]
    if merge_cols:
        merged = pred_df.merge(test_df, on=merge_cols, how="left")
        X_test = merged

        # ---- EJ subgroup definitions (county-level covariates) ----
        def _string_labels(s, missing_label="missing"):
            out = pd.Series(s).astype("string").fillna(missing_label)
            return out.astype(str).values

        # Poverty: prefer explicit poverty rate; otherwise proxy with median_income (low income ~ high poverty).
        if "poverty_rate" in merged.columns:
            merged["ej_poverty"] = _qsplit(merged["poverty_rate"], high_label="high_poverty", low_label="low_poverty")
            subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])
        elif "pct_poverty" in merged.columns:
            merged["ej_poverty"] = _qsplit(merged["pct_poverty"], high_label="high_poverty", low_label="low_poverty")
            subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])
        elif "median_income" in merged.columns:
            merged["ej_poverty"] = _qsplit(merged["median_income"], high_label="high_poverty", low_label="low_poverty", invert=True)
            subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])

        # SVI (if present)
        for svi_col in ["SVI", "svi", "svi_total", "svi_overall", "svi_rank"]:
            if svi_col in merged.columns:
                merged["ej_svi"] = _qsplit(merged[svi_col], high_label="high_SVI", low_label="low_SVI")
                subgroup_columns["ej_svi"] = _string_labels(merged["ej_svi"])
                break

        # Racial majority: person-centered labels (majority_white vs majority_BIPOC), no default to white.
        if "pct_white" in merged.columns:
            pw = pd.to_numeric(merged["pct_white"], errors="coerce")
            merged["ej_racial_majority"] = np.where(pw >= 50, "majority_white", "majority_BIPOC")
            subgroup_columns["ej_racial_majority"] = _string_labels(merged["ej_racial_majority"])

        # Rural vs urban: use NCHS urban-rural (1=most urban ... 6=most rural).
        if "nchs_urban_rural" in merged.columns:
            ur = pd.to_numeric(merged["nchs_urban_rural"], errors="coerce")
            merged["ej_rural"] = np.where(ur >= 5, "rural", "urban")
            subgroup_columns["ej_rural"] = _string_labels(merged["ej_rural"])

        # High vs low pesticide use: total kg per cropland acre when possible.
        if "pesticide_total_kg" in merged.columns:
            denom = pd.to_numeric(merged.get("cropland_acres"), errors="coerce")
            num = pd.to_numeric(merged["pesticide_total_kg"], errors="coerce")
            per_acre = num / denom.replace(0, np.nan)
            merged["ej_pesticide_use"] = _qsplit(per_acre, high_label="high_pesticide_use", low_label="low_pesticide_use")
            subgroup_columns["ej_pesticide_use"] = _string_labels(merged["ej_pesticide_use"])

        # High vs low IPM breadth: prefer per-acre breadth if present.
        if "ipm_breadth_acre" in merged.columns:
            merged["ej_ipm_breadth"] = _qsplit(merged["ipm_breadth_acre"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
            subgroup_columns["ej_ipm_breadth"] = _string_labels(merged["ej_ipm_breadth"])
        elif "ipm_breadth_value" in merged.columns:
            merged["ej_ipm_breadth"] = _qsplit(merged["ipm_breadth_value"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
            subgroup_columns["ej_ipm_breadth"] = _string_labels(merged["ej_ipm_breadth"])

        # Keep state_fips subgroup too (useful context)
        if "state_fips" in merged.columns:
            subgroup_columns["state_fips"] = merged["state_fips"].values

        if "YEAR" in merged.columns:
            time_column = "YEAR"
    print("Merged test rows:", len(merged) if merge_cols else 0)

task = detect_task(y_true)
print("Task detected:", task)


## Run evaluation

In [ ]:
result = evaluate(
    y_true,
    y_pred,
    task=task,
    X_test=X_test,
    subgroup_columns=subgroup_columns if subgroup_columns else None,
    time_column=time_column,
    top_n_errors=20,
    min_subgroup_support=5,
)

print("Task:", result["task_detected"])


## 1. Headline metrics

In [ ]:
display(result["metrics"])


## 2. Core diagnostic plots

## 4b. Map: prediction error by place (county)

Choropleth of **mean absolute error** by county; optionally highlight counties with **most errors** vs **best predictions**.

In [ ]:
# Build per-county error stats from predictions (works even without X_test)
pred_df["abs_residual"] = np.abs(pred_df["actual"] - pred_df["prediction"])
fips5 = lambda s: s.astype(str).str.zfill(5)
county_error = (
    pred_df.groupby(pred_df["FIPS"].astype(str).str.zfill(5))
    .agg(mean_abs_error=("abs_residual", "mean"), n=("abs_residual", "count"))
    .reset_index()
)
county_error["FIPS"] = fips5(county_error["FIPS"].astype(str))

# Optional: label worst vs best (e.g. top/bottom 20% of counties by mean error)
q_high = county_error["mean_abs_error"].quantile(0.80)
q_low = county_error["mean_abs_error"].quantile(0.20)
county_error["error_tier"] = "middle"
county_error.loc[county_error["mean_abs_error"] >= q_high, "error_tier"] = "worst"
county_error.loc[county_error["mean_abs_error"] <= q_low, "error_tier"] = "best"

# Load US counties GeoJSON for choropleth
import urllib.request
import json
import plotly.express as px

COUNTIES_GEOJSON_URL = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
with urllib.request.urlopen(COUNTIES_GEOJSON_URL) as url:
    counties_geojson = json.loads(url.read())

# Map 1: continuous color by mean absolute error (red = high error, green = low)
fig1 = px.choropleth(
    county_error,
    geojson=counties_geojson,
    locations="FIPS",
    color="mean_abs_error",
    scope="usa",
    featureidkey="id",
    color_continuous_scale="RdYlGn_r",  # red = high error, green = low
    title=f"Mean absolute prediction error by county ({TARGET})",
)
fig1.update_geos(fitbounds="locations", visible=False)
fig1.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=450)
fig1.show()

# Map 2: categorical — worst (red), best (green), middle (gray)
fig2 = px.choropleth(
    county_error,
    geojson=counties_geojson,
    locations="FIPS",
    color="error_tier",
    scope="usa",
    featureidkey="id",
    category_orders={"error_tier": ["worst", "middle", "best"]},
    color_discrete_map={"worst": "#e74c3c", "middle": "#bdc3c7", "best": "#2ecc71"},
    title=f"Worst / best prediction counties (top & bottom 20% by mean error) — {TARGET}",
)
fig2.update_geos(fitbounds="locations", visible=False)
fig2.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=450)
fig2.show()


In [ ]:
for fig, title in result["figures"]:
    fig.suptitle(title, y=1.02)
    plt.show()


## 3. Subgroup performance (if available)

In [ ]:
for name, tbl in result["subgroup_tables"].items():
    print(f"Subgroup: {name}")
    display(tbl)
    print()


## D. EJ subgroup analysis

County-level subgroup tables stratified by EJ-relevant dimensions.

**Residual = actual − predicted** (positive => underprediction; negative => overprediction).


In [ ]:
# Build a county-level table (one row per FIPS)
if X_test is None or "FIPS" not in X_test.columns:
    raise ValueError("X_test with FIPS is required for EJ subgroup analysis. Make sure TEST_PATH merges.")

county_pred = (
    pred_df.assign(FIPS=pred_df["FIPS"].astype(str).str.zfill(5))
    .groupby("FIPS")
    .agg(mean_actual=("actual", "mean"), mean_predicted=("prediction", "mean"))
    .reset_index()
)
county_pred["residual"] = county_pred["mean_actual"] - county_pred["mean_predicted"]
county_pred["abs_residual"] = county_pred["residual"].abs()

# Pull (roughly) county-constant covariates from X_test
X_test2 = X_test.copy()
X_test2["FIPS"] = X_test2["FIPS"].astype(str).str.zfill(5)
county_covars = X_test2.sort_values(["FIPS", "YEAR"] if "YEAR" in X_test2.columns else ["FIPS"]).drop_duplicates("FIPS")

county = county_pred.merge(county_covars, on="FIPS", how="left")

# Use population weighting when available (public-health relevant)
weights = county["population"].values if "population" in county.columns else None

ej_group_cols = [c for c in ["ej_poverty", "ej_svi", "ej_racial_majority", "ej_rural", "ej_pesticide_use", "ej_ipm_breadth"] if c in county.columns]
if not ej_group_cols:
    raise ValueError("No EJ subgroup columns found. Check column names in test data (poverty/SVI/rurality/race/IPM/pesticides).")

for col in ej_group_cols:
    tbl = regression_subgroup_stats(
        county["mean_actual"].values,
        county["mean_predicted"].values,
        groups=county[col].values,
        sample_weight=weights,
    ).sort_values("mean_residual", ascending=False)
    print(f"EJ subgroup: {col}")
    display(tbl)
    print()



## E. High-burden county profiling

Compare the **top decile** vs **bottom decile** of counties by *predicted* asthma burden.


In [ ]:
county_rank = county.copy()
county_rank = county_rank.dropna(subset=["mean_predicted"])
q90 = county_rank["mean_predicted"].quantile(0.90)
q10 = county_rank["mean_predicted"].quantile(0.10)
county_rank["burden_group"] = np.where(county_rank["mean_predicted"] >= q90, "top_decile", np.where(county_rank["mean_predicted"] <= q10, "bottom_decile", None))
profile = county_rank[county_rank["burden_group"].notna()].copy()

# Candidate profile columns (only keep those that exist)
candidate_cols = [
    # IPM / documentation
    "ipm_breadth_acre", "ipm_breadth_value", "chemical_reliance_acre", "chemical_reliance_value",
    "ipm_doc_coverage_share", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age",
    "ipm_primary_match_tier",
    # Pesticides (macro)
    "pesticide_total_kg", "pesticide_respiratory_kg",
    # Crop mix / ag
    "pct_cropland", "cropland_acres", "cropland_diversity", "specialty_crop_share", "county_crop_concentration", "total_ag_value",
    "corn_acres", "soybean_acres", "cotton_acres", "wheat_acres", "hay_acres", "fruit_veg_acres", "rice_acres", "sorghum_acres",
    # EJ / demographics
    "median_income", "median_age", "population", "pct_white", "pct_black", "pct_asian", "pct_hispanic", "nchs_urban_rural",
    # Health context
    "CSMOKING", "OBESITY", "DIABETES",
]
profile_cols = [c for c in candidate_cols if c in profile.columns]

prof_tbl = profile.groupby("burden_group")[profile_cols].mean(numeric_only=True).T
if "top_decile" in prof_tbl.columns and "bottom_decile" in prof_tbl.columns:
    prof_tbl["top_minus_bottom"] = prof_tbl["top_decile"] - prof_tbl["bottom_decile"]
display(prof_tbl.sort_values("top_minus_bottom", ascending=False, na_position="last"))



## F. Error analysis through an EJ lens

Ask whether large errors align with social vulnerability or with data/documentation coverage.


In [ ]:
# County-level error (abs_residual) vs documentation / crop specialization / pesticides
err = county.copy()
err = err.dropna(subset=["abs_residual"])

def _safe_corr(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    m = ~(pd.isna(x) | pd.isna(y))
    if m.sum() < 20:
        return np.nan
    return float(np.corrcoef(x[m], y[m])[0,1])

lens_vars = [
    "ipm_doc_coverage_share", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age",
    "specialty_crop_share", "county_crop_concentration",
    "pesticide_total_kg", "pesticide_respiratory_kg",
    "ipm_breadth_acre", "chemical_reliance_acre",
    "median_income", "pct_white", "nchs_urban_rural",
]
lens_vars = [v for v in lens_vars if v in err.columns]

corrs = pd.DataFrame({
    "variable": lens_vars,
    "corr_with_abs_error": [ _safe_corr(err[v], err["abs_residual"]) for v in lens_vars ],
}).sort_values("corr_with_abs_error", ascending=False)
display(corrs)

# Top error counties with EJ-relevant fields
show_cols = [c for c in [
    "FIPS", "NAME", "state_fips", "mean_actual", "mean_predicted", "residual", "abs_residual",
    "ej_poverty", "ej_svi", "ej_racial_majority", "ej_rural", "ej_pesticide_use", "ej_ipm_breadth",
    "ipm_doc_coverage_share", "mean_text_quality", "mean_geo_confidence", "specialty_crop_share",
] if c in err.columns]
display(err.sort_values("abs_residual", ascending=False).head(25)[show_cols])



## 4. Top error inspection

In [ ]:
display(result["top_errors"])


## 5. Interpretation and next steps

In [ ]:
print("Interpretation:")
print(result["interpretation"])
print()
print("Recommended next steps:")
print(result["next_steps"])


## Optional: save figures

In [ ]:
SAVE_FIGURES = False  # set True to write PNGs
if SAVE_FIGURES:
    out_dir = RESULTS_DIR / TARGET / "evaluation"
    out_dir.mkdir(parents=True, exist_ok=True)
    for fig, title in result["figures"]:
        fig.savefig(out_dir / f"{title}.png", dpi=150, bbox_inches="tight")
    print("Saved to", out_dir)


In [ ]:
# --- Raw vs Engineered: subgroup RMSE deltas + significance (Asthma + COPD) ---
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _find_module_dir(start: Path) -> Path:
    # Prefer local modeling/ if we're already inside it.
    if (start / "model_evaluation.py").exists():
        return start
    for parent in [start] + list(start.parents):
        if (parent / "model_evaluation.py").exists():
            return parent
        if (parent / "modeling" / "model_evaluation.py").exists():
            return parent / "modeling"
    raise FileNotFoundError("Could not find model_evaluation.py")


MODULE_DIR = _find_module_dir(Path.cwd().resolve())
REPO_ROOT = MODULE_DIR.parent
sys.path.insert(0, str(MODULE_DIR))

from model_evaluation import regression_subgroup_stats

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "modeling" / "results"

TARGETS_TO_RUN = ["CASTHMA", "COPD"]
EXPOSURE_KEYS = ["Aggs_raw", "Aggs_engineered"]
MODEL_NAME = "xgboost"

MIN_SUBGROUP_SUPPORT = 5
TOP_GROUPS_DISPLAY = 10

RANDOM_STATE = 42
N_PERM = 1000


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).ravel().astype(float)
    y_pred = np.asarray(y_pred).ravel().astype(float)
    m = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[m], y_pred[m]
    if len(y_true) == 0:
        return np.nan
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def _qsplit(series, *, q=0.5, high_label="high", low_label="low", invert=False):
    s = pd.to_numeric(series, errors="coerce")
    thr = s.quantile(q)
    if invert:
        out = np.where(s <= thr, high_label, low_label)
    else:
        out = np.where(s >= thr, high_label, low_label)
    out[pd.isna(s)] = None
    return out


def load_preds(target: str, exposure_key: str):
    pred_path = RESULTS_DIR / target / f"{MODEL_NAME}_predictions_{exposure_key}.csv"
    if not pred_path.exists():
        raise FileNotFoundError(f"Missing predictions file: {pred_path}")
    pred_df = pd.read_csv(pred_path)
    y_true = pred_df["actual"].values
    y_pred = pred_df["prediction"].values
    return pred_df, y_true, y_pred


def build_subgroup_columns(pred_df: pd.DataFrame, target: str):
    # Merge with test data for covariates used to define EJ subgroups.
    test_path = DATA_DIR / f"test_{target}.csv"
    test_df = pd.read_csv(test_path)

    merge_cols = [c for c in ["FIPS", "YEAR"] if c in pred_df.columns and c in test_df.columns]
    if not merge_cols:
        return {}, None

    merged = pred_df.merge(test_df, on=merge_cols, how="left")

    subgroup_columns = {}

    def _string_labels(s, missing_label="missing"):
        out = pd.Series(s).astype("string").fillna(missing_label)
        return out.astype(str).values

    # Poverty: prefer explicit poverty_rate; otherwise proxy with median_income.
    if "poverty_rate" in merged.columns:
        merged["ej_poverty"] = _qsplit(merged["poverty_rate"], high_label="high_poverty", low_label="low_poverty")
        subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])
    elif "pct_poverty" in merged.columns:
        merged["ej_poverty"] = _qsplit(merged["pct_poverty"], high_label="high_poverty", low_label="low_poverty")
        subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])
    elif "median_income" in merged.columns:
        merged["ej_poverty"] = _qsplit(merged["median_income"], high_label="high_poverty", low_label="low_poverty", invert=True)
        subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])

    # SVI
    for svi_col in ["SVI", "svi", "svi_total", "svi_overall", "svi_rank"]:
        if svi_col in merged.columns:
            merged["ej_svi"] = _qsplit(merged[svi_col], high_label="high_SVI", low_label="low_SVI")
            subgroup_columns["ej_svi"] = _string_labels(merged["ej_svi"])
            break

    # Racial majority: person-centered labels (majority_white vs majority_BIPOC), no default to white.
    if "pct_white" in merged.columns:
        pw = pd.to_numeric(merged["pct_white"], errors="coerce")
        merged["ej_racial_majority"] = np.where(pw >= 50, "majority_white", "majority_BIPOC")
        subgroup_columns["ej_racial_majority"] = _string_labels(merged["ej_racial_majority"])

    # Rural/urban proxy via nchs_urban_rural.
    if "nchs_urban_rural" in merged.columns:
        ur = pd.to_numeric(merged["nchs_urban_rural"], errors="coerce")
        merged["ej_rural"] = np.where(ur >= 5, "rural", "urban")
        subgroup_columns["ej_rural"] = _string_labels(merged["ej_rural"])

    # Pesticide use proxy via pesticide_total_kg per cropland acre.
    if "pesticide_total_kg" in merged.columns:
        denom = pd.to_numeric(merged.get("cropland_acres"), errors="coerce")
        num = pd.to_numeric(merged["pesticide_total_kg"], errors="coerce")
        per_acre = num / denom.replace(0, np.nan)
        merged["ej_pesticide_use"] = _qsplit(per_acre, high_label="high_pesticide_use", low_label="low_pesticide_use")
        subgroup_columns["ej_pesticide_use"] = _string_labels(merged["ej_pesticide_use"])

    # IPM breadth
    if "ipm_breadth_acre" in merged.columns:
        merged["ej_ipm_breadth"] = _qsplit(merged["ipm_breadth_acre"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
        subgroup_columns["ej_ipm_breadth"] = _string_labels(merged["ej_ipm_breadth"])
    elif "ipm_breadth_value" in merged.columns:
        merged["ej_ipm_breadth"] = _qsplit(merged["ipm_breadth_value"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
        subgroup_columns["ej_ipm_breadth"] = _string_labels(merged["ej_ipm_breadth"])

    if "state_fips" in merged.columns:
        subgroup_columns["state_fips"] = merged["state_fips"].values

    return subgroup_columns, merged


def significance_paired_squared_error_diff(y_true, y_raw, y_eng, groups, group_value):
    # Paired per-record squared error difference: (eng - raw)
    y_true = np.asarray(y_true).ravel().astype(float)
    y_raw = np.asarray(y_raw).ravel().astype(float)
    y_eng = np.asarray(y_eng).ravel().astype(float)

    groups = pd.Series(groups).astype("string").fillna("missing").astype(str).to_numpy().ravel()

    valid = ~(np.isnan(y_true) | np.isnan(y_raw) | np.isnan(y_eng))
    mask = valid & (groups == str(group_value))
    if mask.sum() < MIN_SUBGROUP_SUPPORT:
        return np.nan, int(mask.sum())

    diff = (y_true[mask] - y_eng[mask]) ** 2 - (y_true[mask] - y_raw[mask]) ** 2
    if np.allclose(diff, 0):
        return 1.0, int(mask.sum())

    # Prefer Wilcoxon if scipy exists.
    try:
        from scipy.stats import wilcoxon
        stat = wilcoxon(diff, alternative="two-sided")
        return float(stat.pvalue), int(mask.sum())
    except Exception:
        # Permutation sign test (flip signs of paired diffs).
        rng = np.random.default_rng(RANDOM_STATE)
        obs = float(np.mean(diff))
        # B x n matrix of +/-1 signs
        signs = rng.choice([-1.0, 1.0], size=(N_PERM, diff.shape[0]))
        perm_means = (signs * diff[None, :]).mean(axis=1)
        p = float((np.abs(perm_means) >= abs(obs)).mean())
        return p, int(mask.sum())


for target in TARGETS_TO_RUN:
    print("\n===============================")
    print(f"Target: {target}")

    pred_raw_df, y_true, y_pred_raw = load_preds(target, EXPOSURE_KEYS[0])
    _, _, y_pred_eng = load_preds(target, EXPOSURE_KEYS[1])

    print(f"Overall RMSE raw ({EXPOSURE_KEYS[0]}): {rmse(y_true, y_pred_raw):.4f}")
    print(f"Overall RMSE engineered ({EXPOSURE_KEYS[1]}): {rmse(y_true, y_pred_eng):.4f}")

    subgroup_columns, merged = build_subgroup_columns(pred_raw_df, target)
    if not subgroup_columns:
        print("No subgroup columns available from merged test data; skipping subgroup delta plots.")
        continue

    # Compute subgroup RMSE deltas
    for subgroup_name, groups in subgroup_columns.items():
        stats_raw = regression_subgroup_stats(y_true, y_pred_raw, groups)
        stats_eng = regression_subgroup_stats(y_true, y_pred_eng, groups)

        common_groups = stats_raw.index.intersection(stats_eng.index)
        if len(common_groups) == 0:
            continue

        rmse_raw = stats_raw.loc[common_groups, "RMSE"]
        rmse_eng = stats_eng.loc[common_groups, "RMSE"]
        n_vals = stats_raw.loc[common_groups, "n"] if "n" in stats_raw.columns else pd.Series(index=common_groups, data=np.nan)

        delta = rmse_eng - rmse_raw
        ok = n_vals >= MIN_SUBGROUP_SUPPORT
        delta = delta[ok].dropna()

        if len(delta) == 0:
            continue

        # Limit plot to the groups with largest absolute deltas.
        show_groups = delta.reindex(delta.abs().sort_values(ascending=False).head(TOP_GROUPS_DISPLAY).index)
        show_groups = show_groups.sort_values()  # for readability in barh

        fig, ax = plt.subplots(figsize=(10, 0.35 * len(show_groups) + 2.5))
        y = np.arange(len(show_groups))
        ax.barh(y, show_groups.values, color="steelblue")
        ax.set_yticks(y)
        ax.set_yticklabels(show_groups.index.astype(str))
        ax.axvline(0, color="black", lw=1)
        ax.set_title(f"{target}: RMSE delta (engineered - raw) by {subgroup_name}")
        ax.set_xlabel("RMSE_engineered - RMSE_raw")

        for yi, val in enumerate(show_groups.values):
            ax.text(val, yi, f"{val:+.3f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)

        plt.tight_layout()
        plt.show()

        # Significance tests for all (not just displayed) groups with enough support.
        results_rows = []
        for g in delta.index:
            p_value, n_g = significance_paired_squared_error_diff(y_true, y_pred_raw, y_pred_eng, groups, g)
            results_rows.append({
                "group": str(g),
                "n": int(n_g),
                f"rmse_{EXPOSURE_KEYS[0]}": float(rmse_raw.loc[g]),
                f"rmse_{EXPOSURE_KEYS[1]}": float(rmse_eng.loc[g]),
                "rmse_delta": float(delta.loc[g]),
                "p_value": p_value,
            })

        sig_df = pd.DataFrame(results_rows).sort_values("p_value")
        display(sig_df.head(10))

